# Secret Spot: Surf Forecasting via Graph-Based Wave Refraction

Prototype implementation for Part 3.


## Imports and constants


In [1]:
import heapq
import math

import numpy as np
import matplotlib.pyplot as plt

G = 9.81  # gravitational acceleration, m/s^2


## Shared conventions

```
GRID:        depth_grid[row][col], depths in meters
             water = positive depth; land = 0 or negative
             row 0 = north edge, col 0 = west edge
CELL:        a (row, col) pair
CELL SIZE:   cell_size = (dy, dx) in meters (north-south, east-west)
DIRECTIONS:  degrees clockwise from north; swell direction = where the swell comes FROM
             (same convention buoys and Surfline use)
```


## Physics reference

Each equation is shown as it's written in physics, followed by how it appears in the code.

| Symbol | Meaning | Code name | Units |
|---|---|---|---|
| $g$ | gravitational acceleration | `G` | m/s² |
| $T$ | wave period | `period` | s |
| $h$ | water depth | `depth` | m |
| $\omega$ | angular frequency | `angular_frequency` | rad/s |
| $k$ | wavenumber (2π ÷ wavelength) | `wavenumber` | rad/m |
| $c$ | wave speed | `speed` | m/s |

### 1. Angular frequency (`wave_speed`)

$$\omega = \frac{2\pi}{T}$$

```python
angular_frequency = 2 * math.pi / period
```

### 2. Dispersion relation: the core wave equation (`wave_speed`)

Links a wave's period to its wavenumber at a given depth. It can't be solved for $k$ directly, so the code solves it numerically.

$$\omega^2 = g\,k\,\tanh(k\,h)$$

```python
residual = G * wavenumber * tanh_term - angular_frequency**2   # zero when the equation holds
```

### 3. Wave speed (`wave_speed`)

$$c = \frac{\omega}{k}$$

```python
return angular_frequency / wavenumber
```

### 4. Newton's method for solving the dispersion relation (`wave_speed`)

Rewrite the equation as $f(k) = 0$ and improve the guess repeatedly:

$$f(k) = g\,k\,\tanh(kh) - \omega^2 \qquad f'(k) = g\,\tanh(kh) + g\,k\,h\,\left(1 - \tanh^2(kh)\right)$$

$$k_{\text{new}} = k - \frac{f(k)}{f'(k)}$$

```python
residual = G * wavenumber * tanh_term - angular_frequency**2
residual_slope = G * tanh_term + G * wavenumber * depth * (1 - tanh_term**2)
new_wavenumber = wavenumber - residual / residual_slope
```

Starting guess, a common approximation that's close at any depth:

$$k_0 = \frac{\omega^2}{g} \qquad k \approx \frac{k_0}{\sqrt{\tanh(k_0 h)}}$$

```python
deep_wavenumber = angular_frequency**2 / G
wavenumber = deep_wavenumber / math.sqrt(math.tanh(deep_wavenumber * depth))
```

### 5. Limits used as tests (`wave_speed` tests)

Deep water ($\tanh(kh) \to 1$):
$$c_{\text{deep}} = \frac{g\,T}{2\pi}$$
```python
deep_limit = G * test_period / (2 * math.pi)
```

Shallow water ($\tanh(kh) \to kh$):
$$c_{\text{shallow}} = \sqrt{g\,h}$$
```python
shallow_limit = math.sqrt(G * 0.5)
```

### 6. Travel time across one edge (`build_graph`)

Distance between two cell centers, with cell size $(\Delta y, \Delta x)$ and steps of $\Delta r$ rows and $\Delta c$ columns:

$$\Delta s = \sqrt{(\Delta r\,\Delta y)^2 + (\Delta c\,\Delta x)^2}$$

Travel time, averaging the "slowness" ($1/c$) of the two cells:

$$\Delta t = \frac{\Delta s}{2}\left(\frac{1}{c_1} + \frac{1}{c_2}\right)$$

```python
distance = math.sqrt((row_step * dy)**2 + (col_step * dx)**2)
travel_time = (distance / 2) * (1 / speed_1 + 1 / speed_2)
```

### 7. Converting degrees to meters (`load_bathymetry`)

$$\Delta y = \Delta\text{lat} \times 111{,}000 \qquad \Delta x = \Delta\text{lon} \times 111{,}000 \times \cos(\text{latitude})$$

```python
dy = lat_spacing * METERS_PER_DEGREE
dx = lon_spacing * METERS_PER_DEGREE * math.cos(math.radians(mean_latitude))
```

### 8. Staggered start times along the wave front (`get_source_cells`)

The swell travels toward bearing $\beta$ = swell direction + 180°. A cell's position along that direction, and its start time with reference speed $c_{\text{ref}}$:

$$p = y_{\text{north}}\cos\beta + x_{\text{east}}\sin\beta \qquad t_{\text{start}} = \frac{p - p_{\min}}{c_{\text{ref}}}$$

```python
position_along_travel = north_position * travel_north + east_position * travel_east
start_time = (position_along_travel - earliest) / reference_speed
```


# 1. Grid building


In [2]:
def make_synthetic_grid(shape, rows, cols, base_depth, cell_size):
    """
    Build a synthetic seafloor for testing.

    Inputs:
        shape (str): "flat", "mound", or "slope"
        rows, cols (int): grid size, at least 1 each ("slope" needs at least 2 columns)
        base_depth (float): depth of open water in meters (positive)
        cell_size (tuple): (dy, dx) in meters; not used to build the grid, but kept
                           so every grid-building function shares the same inputs
    Returns:
        depth_grid (np.ndarray): rows x cols array of depths in meters
                                 (water > 0, land <= 0)
    Assumptions:
        Row 0 is the north edge and column 0 is the west edge.
        "mound" is centered in the grid; "slope" has its beach (depth 0) on the east edge.
    """
    if rows < 1 or cols < 1:
        raise ValueError("grid must have at least one row and column")

    depth_grid = np.full((rows, cols), float(base_depth))

    if shape == "flat":
        return depth_grid

    elif shape == "mound":
        center_row, center_col = rows // 2, cols // 2
        radius = min(rows, cols) // 4
        peak_depth = base_depth / 4
        for r in range(rows):
            for c in range(cols):
                distance_from_center = math.hypot(r - center_row, c - center_col)
                if distance_from_center < radius:
                    # shallowest at the center, back to base_depth at the edge
                    depth_grid[r, c] = peak_depth + (base_depth - peak_depth) * (distance_from_center / radius)
        return depth_grid

    elif shape == "slope":
        if cols < 2:
            raise ValueError("slope needs at least 2 columns")
        for c in range(cols):
            depth_grid[:, c] = base_depth * (1 - c / (cols - 1))
        return depth_grid  # last column is 0 = land (the beach)

    else:
        raise ValueError(f"unknown shape: {shape!r}")

In [3]:
# Test: make_synthetic_grid
flat = make_synthetic_grid("flat", 4, 5, 20.0, (10, 10))
assert flat.shape == (4, 5), "flat grid has wrong shape"
assert np.all(flat == 20.0), "flat grid should be 20 m everywhere"

mound = make_synthetic_grid("mound", 20, 20, 20.0, (10, 10))
assert mound[10, 10] == 5.0, "mound center should be base_depth / 4"
assert mound[0, 0] == 20.0, "corner should be untouched base_depth"
assert mound.min() == 5.0 and mound.max() == 20.0, "mound depths out of range"

slope = make_synthetic_grid("slope", 3, 5, 20.0, (10, 10))
assert np.all(slope[:, 0] == 20.0), "west edge should be base_depth"
assert np.all(slope[:, -1] == 0.0), "east edge should be 0 (beach)"
assert np.all(np.diff(slope[0]) < 0), "depth should decrease toward the east"

# invalid inputs should raise errors
for bad_call in [lambda: make_synthetic_grid("flat", 0, 5, 20, (10, 10)),
                 lambda: make_synthetic_grid("reef", 5, 5, 20, (10, 10)),
                 lambda: make_synthetic_grid("slope", 5, 1, 20, (10, 10))]:
    try:
        bad_call()
        raise AssertionError("expected a ValueError")
    except ValueError:
        pass

print("make_synthetic_grid: all tests passed")

make_synthetic_grid: all tests passed


In [4]:
def load_bathymetry(file_path, bounds):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: file_path (string), bounds = (south, north, west, east) in degrees
    # Returns: depth_grid (2D array), cell_size (dy, dx) in meters
    #
    # read elevation values, latitudes, and longitudes from file
    # keep only points inside bounds
    #
    # if no points remain:
    #     raise error "bounds contain no data"
    #
    # GEBCO and NOAA store ELEVATION: negative below sea level
    # depth_grid = -elevation                    # flip sign so water depth is positive
    #
    # for each cell with a missing value:
    #     set depth to 0                         # treat as land, so no wave path crosses it
    #     record a warning with the count of missing cells
    #
    # convert grid spacing from degrees to meters
    # dy = latitude spacing  * 111,000
    # dx = longitude spacing * 111,000 * cos(average latitude)
    #
    # return depth_grid, (dy, dx)
    raise NotImplementedError


In [5]:
# Test: load_bathymetry
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [6]:
def wave_speed(depth, period):
    """
    Speed of a wave of a given period at a given depth.

    Inputs:
        depth (float): water depth in meters (land if <= 0)
        period (float): wave period in seconds (must be > 0)
    Returns:
        speed (float) in m/s, or None for land
    Assumptions:
        Uses the full wave speed (dispersion) equation,
        angular_frequency^2 = g * wavenumber * tanh(wavenumber * depth),
        solved for the wavenumber with Newton's method, so it works in deep,
        intermediate, and shallow water.
    """
    if period <= 0:
        raise ValueError("period must be positive")
    if depth <= 0:
        return None

    angular_frequency = 2 * math.pi / period
    deep_wavenumber = angular_frequency**2 / G
    # good starting guess at any depth
    wavenumber = deep_wavenumber / math.sqrt(math.tanh(deep_wavenumber * depth))

    for _ in range(50):
        tanh_term = math.tanh(wavenumber * depth)
        residual = G * wavenumber * tanh_term - angular_frequency**2        # how far off the guess is
        residual_slope = G * tanh_term + G * wavenumber * depth * (1 - tanh_term**2)
        new_wavenumber = wavenumber - residual / residual_slope              # Newton step
        if abs(new_wavenumber - wavenumber) < 1e-12 * wavenumber:
            wavenumber = new_wavenumber
            break
        wavenumber = new_wavenumber
    else:
        raise RuntimeError("wave_speed did not converge")

    return angular_frequency / wavenumber   # speed = angular frequency / wavenumber

In [7]:
# Test: wave_speed
test_period = 10.0

# deep water should match g * period / (2 * pi)
deep = wave_speed(1000, test_period)
deep_limit = G * test_period / (2 * math.pi)
assert abs(deep - deep_limit) / deep_limit < 0.001, f"deep: got {deep:.3f}, expected {deep_limit:.3f}"

# very shallow water should match sqrt(g * depth)
shallow = wave_speed(0.5, test_period)
shallow_limit = math.sqrt(G * 0.5)
assert abs(shallow - shallow_limit) / shallow_limit < 0.01, f"shallow: got {shallow:.3f}, expected {shallow_limit:.3f}"

# speed should increase with depth
speeds = [wave_speed(test_depth, test_period) for test_depth in [1, 5, 10, 20, 50, 100]]
assert all(slower < faster for slower, faster in zip(speeds, speeds[1:])), "speed should increase with depth"

# the answer should satisfy the wave equation at many depths and periods
for test_depth in [0.01, 0.1, 1, 5, 20, 49, 200, 5000]:
    for period in [2, 6, 10, 15, 25]:
        angular_frequency = 2 * math.pi / period
        wavenumber = angular_frequency / wave_speed(test_depth, period)
        residual = G * wavenumber * math.tanh(wavenumber * test_depth) - angular_frequency**2
        assert abs(residual) < 1e-9, f"failed at depth {test_depth}, period {period}"

# land and bad input
assert wave_speed(0, test_period) is None and wave_speed(-5, test_period) is None, "land should return None"
try:
    wave_speed(10, 0)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass

print("wave_speed: all tests passed")
print(f"  10 s swell: 1000 m -> {deep:.2f} m/s, 49 m -> {wave_speed(49, test_period):.2f} m/s, 0.5 m -> {shallow:.2f} m/s")

wave_speed: all tests passed
  10 s swell: 1000 m -> 15.61 m/s, 49 m -> 15.09 m/s, 0.5 m -> 2.21 m/s


In [8]:
def build_graph(depth_grid, cell_size, period, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid (2D array), cell_size (dy, dx), period in seconds,
    #        neighbors (8 or 16)
    # Returns: graph (dictionary: cell -> list of (neighbor_cell, travel_time))
    #
    # if neighbors == 8:
    #     offsets = the 8 surrounding cells (up, down, left, right, 4 diagonals)
    # else if neighbors == 16:
    #     offsets = those 8 plus 8 "knight moves" (e.g., 1 down, 2 right)
    # else:
    #     raise error "neighbors must be 8 or 16"
    #
    # compute each cell's speed once, not once per edge
    # speed_grid = wave_speed(depth, period) for every cell
    #
    # graph = empty dictionary
    # for each cell (r, c):
    #     if speed_grid[r][c] is NONE:
    #         continue                           # land cells get no node
    #     graph[(r, c)] = empty list
    #
    #     for each offset (dr, dc):
    #         (nr, nc) = (r + dr, c + dc)
    #         if (nr, nc) is outside the grid or speed_grid[nr][nc] is NONE:
    #             continue
    #         if the move passes through or corners past a land cell:
    #             continue                       # EDGE CASE: no shortcuts through land
    #
    #         distance = sqrt((dr * dy)^2 + (dc * dx)^2)
    #         # average "slowness" of the two cells, so slow shallow cells count fully
    #         travel_time = (distance / 2) * (1 / speed_grid[r][c] + 1 / speed_grid[nr][nc])
    #         append ((nr, nc), travel_time) to graph[(r, c)]
    #
    # return graph
    raise NotImplementedError


In [9]:
# Test: build_graph
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 2. Refraction (Dijkstra)


In [10]:
def get_source_cells(depth_grid, cell_size, period, swell_direction):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid (2D array), cell_size (dy, dx), period in seconds,
    #        swell_direction in degrees (where swell comes FROM)
    # Returns: sources (dictionary: cell -> start_time in seconds);
    #          empty if no water edge faces the swell
    #
    # travel_bearing = (swell_direction + 180) mod 360   # swell travels AWAY from where it comes from
    # travel_north = cos(travel_bearing)
    # travel_east  = sin(travel_bearing)
    #
    # find edge cells where the swell enters the grid
    # candidates = empty set
    # if travel_north < 0: add every cell in the top row (north edge)       # swell heading south
    # if travel_north > 0: add every cell in the bottom row (south edge)    # swell heading north
    # if travel_east  > 0: add every cell in the left column (west edge)    # swell heading east
    # if travel_east  < 0: add every cell in the right column (east edge)   # swell heading west
    # remove land cells from candidates
    #
    # if candidates is empty:
    #     return empty dictionary                # EDGE CASE: swell comes from the land side
    #
    # stagger start times along the wave front
    # reference_speed = average wave_speed(depth, period) over candidates
    # for each cell (r, c) in candidates:
    #     north_position = -r * dy               # rows increase southward
    #     east_position  =  c * dx
    #     position_along_travel[cell] = north_position * travel_north + east_position * travel_east
    #
    # earliest = minimum of position_along_travel
    # sources = empty dictionary
    # for each cell in candidates:
    #     sources[cell] = (position_along_travel[cell] - earliest) / reference_speed
    #
    # return sources
    raise NotImplementedError


In [11]:
# Test: get_source_cells
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [12]:
def dijkstra(graph, sources):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: graph (cell -> list of (neighbor, travel_time)),
    #        sources (cell -> start_time)
    # Returns: travel_times (cell -> fastest arrival time, INFINITY if unreached),
    #          predecessors (cell -> previous cell on the fastest path, NONE for sources)
    #
    # --- A. Initialization ---
    # for each cell in graph:
    #     travel_times[cell] = INFINITY
    #     predecessors[cell] = NONE
    # finalized = empty set
    # queue = empty priority queue               # always returns the smallest time first
    #
    # --- B. Load the starting cells ---
    # for each (cell, start_time) in sources:
    #     if cell not in graph:
    #         continue                           # skip sources that are land
    #     travel_times[cell] = start_time
    #     add (start_time, cell) to queue
    #
    # --- C. Main loop ---
    # while queue is not empty:
    #     (time, current) = remove smallest from queue      # C1. extract-min
    #     if current in finalized:
    #         continue                                      # C2. skip outdated queue entries
    #     add current to finalized                          # C3. finalize
    #
    #     for each (neighbor, edge_time) in graph[current]: # C4. relaxation
    #         if neighbor in finalized:
    #             continue
    #         new_time = travel_times[current] + edge_time
    #         if new_time < travel_times[neighbor]:
    #             travel_times[neighbor] = new_time
    #             predecessors[neighbor] = current
    #             add (new_time, neighbor) to queue
    #
    # --- D. Termination ---
    # queue is empty: every reachable cell is finalized;
    # cells still at INFINITY were never reached (blocked by land)
    #
    # return travel_times, predecessors
    raise NotImplementedError


In [13]:
# Test: dijkstra
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [14]:
def reconstruct_path(predecessors, travel_times, cell):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: predecessors and travel_times (from dijkstra), cell (row, col)
    # Returns: path (list of cells from offshore source to cell);
    #          empty list if the cell was never reached
    #
    # --- E. Path reconstruction ---
    # if cell not in travel_times or travel_times[cell] == INFINITY:
    #     return empty list                      # EDGE CASE: unreachable cell
    #
    # path = [cell]
    # while predecessors[path's last cell] is not NONE:
    #     append predecessors[path's last cell] to path
    #     if path is longer than the number of cells in the graph:
    #         raise error "cycle in predecessors"   # safety check: should never happen
    #
    # reverse path                               # now reads source -> cell
    # return path
    raise NotImplementedError


In [15]:
# Test: reconstruct_path
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [16]:
def count_converging_paths(predecessors, travel_times, pin, radius):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: predecessors and travel_times (from dijkstra),
    #        pin (row, col), radius in cells (int)
    # Returns: count (int) of distinct offshore source cells whose paths end near the pin
    #
    # nearby_water = every cell in travel_times within radius of pin
    #
    # if nearby_water is empty:
    #     raise error "no water near pin: move the pin or increase radius"   # EDGE CASE: pin on land
    #
    # feeding_sources = empty set
    # for each cell in nearby_water:
    #     path = reconstruct_path(predecessors, travel_times, cell)
    #     if path is not empty:
    #         add path[0] to feeding_sources     # path[0] = the offshore starting cell
    #
    # return size of feeding_sources             # 0 means fully blocked from this direction
    raise NotImplementedError


In [17]:
# Test: count_converging_paths
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [18]:
def swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid, cell_size, pin, period, directions (list of degrees,
    #        i.e., 0, 10, ..., 350), radius (int), neighbors (8 or 16)
    # Returns: profile (dictionary: direction -> converging path count)
    #
    # graph = build_graph(depth_grid, cell_size, period, neighbors)   # built ONCE
    #
    # profile = empty dictionary
    # for each direction in directions:
    #     sources = get_source_cells(depth_grid, cell_size, period, direction)
    #     if sources is empty:
    #         profile[direction] = 0             # swell from the land side: blocked
    #         continue
    #     travel_times, predecessors = dijkstra(graph, sources)
    #     profile[direction] = count_converging_paths(predecessors, travel_times, pin, radius)
    #
    # return profile
    raise NotImplementedError


In [19]:
# Test: swell_window_profile
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 3. Scoring


In [20]:
def angle_difference(a, b):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: two directions in degrees (float)
    # Returns: smallest angle between them, 0 to 180 (float)
    #
    # diff = |a - b| mod 360
    # if diff > 180:
    #     diff = 360 - diff                      # EDGE CASE: wrap around north
    # return diff
    raise NotImplementedError


In [21]:
# Test: angle_difference
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [22]:
def score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: profile (direction -> path count, from swell_window_profile),
    #        buoy_reading (peak_energy in m²/Hz, peak_period in s, peak_direction in degrees, height in ft),
    #        wind (speed in knots, direction in degrees, where it comes FROM),
    #        offshore_direction in degrees (wind direction that blows offshore at this spot),
    #        cutoffs (energy_min, period_min, exposure_min, light_wind_max, offshore_tolerance)
    # Returns: verdict ("worth checking", "not worth it", or "not enough data"),
    #          reasons (list of strings), swell numbers for display (or NONE)
    #
    # if any buoy_reading or wind value is missing:
    #     return "not enough data", ["missing buoy or wind data"], NONE   # EDGE CASE
    #
    # reasons = empty list
    #
    # --- Is there something? ---
    # nearest_direction = buoy peak_direction rounded to the nearest 10 degrees (360 -> 0)
    # exposed = profile[nearest_direction] >= cutoffs.exposure_min
    # enough_energy = buoy peak_energy >= cutoffs.energy_min
    # long_enough = buoy peak_period >= cutoffs.period_min
    # something = exposed and enough_energy and long_enough
    # for each check that failed: add an explanation to reasons
    #     # i.e., "spot is blocked from this swell direction", "low wave energy"
    #
    # --- Is it clean? ---
    # light_wind = wind speed <= cutoffs.light_wind_max
    # offshore = angle_difference(wind direction, offshore_direction) <= cutoffs.offshore_tolerance
    # clean = light_wind or offshore
    # if not clean: add "onshore or cross-shore wind" to reasons
    #
    # if something and clean:
    #     verdict = "worth checking"
    # else:
    #     verdict = "not worth it"
    #
    # return verdict, reasons, (height, peak_period, peak_direction)
    raise NotImplementedError


In [23]:
# Test: score_conditions
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 4. Full run


In [24]:
def run_forecast(pin, bathymetry_file, bounds, buoy_reading, wind, offshore_direction, cutoffs, radius, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: pin (row, col), bathymetry_file (string), bounds (south, north, west, east),
    #        buoy_reading, wind, offshore_direction, cutoffs (as in score_conditions),
    #        radius (int), neighbors (8 or 16)
    # Returns: report (verdict, reasons, swell numbers, swell window profile)
    #
    # depth_grid, cell_size = load_bathymetry(bathymetry_file, bounds)
    #
    # if pin is outside the grid:
    #     raise error "pin is outside the bathymetry bounds"           # EDGE CASE
    #
    # period = buoy_reading.peak_period
    # directions = [0, 10, 20, ..., 350]
    #
    # profile = swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors)
    # verdict, reasons, swell_numbers = score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs)
    #
    # return report containing verdict, reasons, swell_numbers, profile
    raise NotImplementedError


In [25]:
# Test: run_forecast
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)
